# SO3.2-03 — Inventário da população WorldPop desagregada por sexo

## Objetivo

Identificar e validar a fonte de população desagregada por sexo que será utilizada no SO3.2, mantendo compatibilidade com a série WorldPop adotada como dado padrão no PRAIS 4.

Esta etapa verifica:

- a disponibilidade temporal da série WorldPop Global1 2000–2020;
- a estrutura dos arquivos anuais para o Brasil;
- a presença das categorias feminina e masculina;
- a presença de todas as classes etárias esperadas;
- a correspondência entre o arquivo histórico WorldPop e a coleção de idade/sexo disponível no Google Earth Engine para 2020;
- as propriedades básicas da grade disponível no Earth Engine em 2020.

Esta etapa **não baixa os GeoTIFFs históricos**, não agrega ainda as classes etárias em população feminina/masculina e não executa o cruzamento com a seca.

## Decisão de fonte

O PRAIS 4 utiliza como dado padrão a série WorldPop 2000–2020 e replica 2020 para 2021, 2022 e 2023. Embora exista uma geração WorldPop mais recente para 2015–2030, ela constitui uma fonte alternativa em relação ao dado padrão do processo de reporte de 2026.

Para preservar a coerência com a população total já inventariada no SO3.2-02, este notebook avalia o **WorldPop Global1 2000–2020**.

## 1. Configuração do ambiente

O inventário histórico é feito a partir dos diretórios públicos do WorldPop. O Google Earth Engine é utilizado apenas para verificar a estrutura da coleção de idade e sexo disponível para 2020.

In [1]:
from google.colab import drive
from pathlib import Path
from urllib.parse import urljoin
import re

import ee
import pandas as pd
import requests

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/Cemaden")
PROJECT_ROOT = DRIVE_ROOT / "PRAIS4_SO3_BR"
SO32_ROOT = PROJECT_ROOT / "SO3.2"
SO32_LOGS = SO32_ROOT / "logs"

EE_PROJECT = "cursoqueimadas-503722"

WORLDPOP_ARCHIVE = (
    "https://data.worldpop.org/GIS/AgeSex_structures/"
    "Global_2000_2020"
)

WORLDPOP_GEE_AGESEX = "WorldPop/GP/100m/pop_age_sex"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print(f"Earth Engine inicializado: {EE_PROJECT}")
print(f"Diretório de logs: {SO32_LOGS}")
print(f"Disponível: {SO32_LOGS.is_dir()}")

Mounted at /content/drive
Earth Engine inicializado: cursoqueimadas-503722
Diretório de logs: /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs
Disponível: True


## 2. Estrutura esperada do produto histórico

Para cada ano são esperadas duas categorias de sexo:

- `f` — female;
- `m` — male.

As classes etárias do produto histórico são:

`0, 1, 5, 10, 15, ..., 80`

Isso corresponde a 18 classes por sexo e **36 GeoTIFFs por ano**.

Padrão esperado para o Brasil:

`bra_{sexo}_{classe_etaria}_{ano}.tif`

In [2]:
EXPECTED_YEARS = list(range(2000, 2021))
EXPECTED_SEXES = ["f", "m"]
EXPECTED_AGE_CLASSES = [
    0, 1, 5, 10, 15, 20, 25, 30, 35,
    40, 45, 50, 55, 60, 65, 70, 75, 80
]

EXPECTED_FILES_PER_SEX_YEAR = len(EXPECTED_AGE_CLASSES)
EXPECTED_FILES_PER_YEAR = len(EXPECTED_SEXES) * EXPECTED_FILES_PER_SEX_YEAR
EXPECTED_TOTAL_FILES = len(EXPECTED_YEARS) * EXPECTED_FILES_PER_YEAR

print(f"Anos esperados           : {len(EXPECTED_YEARS)}")
print(f"Classes etárias por sexo : {EXPECTED_FILES_PER_SEX_YEAR}")
print(f"Arquivos esperados/ano   : {EXPECTED_FILES_PER_YEAR}")
print(f"Arquivos esperados total : {EXPECTED_TOTAL_FILES}")

Anos esperados           : 21
Classes etárias por sexo : 18
Arquivos esperados/ano   : 36
Arquivos esperados total : 756


## 3. Inventário remoto dos arquivos 2000–2020

Os diretórios anuais do WorldPop são consultados sem baixar os GeoTIFFs.

In [3]:
session = requests.Session()
session.headers.update({
    "User-Agent": "PRAIS4-SO3.2-Brazil/1.0 (technical inventory; no raster download)"
})

age_pattern = "|".join(str(age) for age in EXPECTED_AGE_CLASSES)

file_pattern = re.compile(
    rf"^bra_([fm])_({age_pattern})_(\d{{4}})\.tif$",
    flags=re.IGNORECASE
)

archive_records = []
directory_status = []

for year in EXPECTED_YEARS:
    year_url = f"{WORLDPOP_ARCHIVE}/{year}/BRA/"
    response = session.get(year_url, timeout=60)

    directory_status.append({
        "year": year,
        "url": year_url,
        "http_status": response.status_code,
    })

    if response.status_code != 200:
        continue

    hrefs = re.findall(
        r'href=["\']([^"\']+)["\']',
        response.text,
        flags=re.IGNORECASE
    )

    filenames = sorted({
        href.split("/")[-1]
        for href in hrefs
        if href.lower().endswith(".tif")
    })

    for filename in filenames:
        match = file_pattern.match(filename)
        if not match:
            continue

        archive_records.append({
            "year": int(match.group(3)),
            "sex": match.group(1).lower(),
            "age_class": int(match.group(2)),
            "filename": filename,
            "url": urljoin(year_url, filename),
        })

directory_status = pd.DataFrame(directory_status)

archive_inventory = (
    pd.DataFrame(
        archive_records,
        columns=["year", "sex", "age_class", "filename", "url"]
    )
    .sort_values(["year", "sex", "age_class"])
    .reset_index(drop=True)
)

print(
    "Diretórios anuais acessíveis:",
    int((directory_status["http_status"] == 200).sum()),
    "de",
    len(EXPECTED_YEARS)
)

print(
    "GeoTIFFs compatíveis com o padrão:",
    len(archive_inventory)
)

Diretórios anuais acessíveis: 21 de 21
GeoTIFFs compatíveis com o padrão: 756


## 4. Completude temporal, por sexo e por classe etária

São verificadas separadamente as quantidades de arquivos femininos e masculinos e o número de classes etárias distintas por ano.

In [4]:

files_by_sex_year = (
    archive_inventory
    .groupby(["year", "sex"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=EXPECTED_YEARS, columns=EXPECTED_SEXES, fill_value=0)
)

age_classes_by_sex_year = (
    archive_inventory
    .groupby(["year", "sex"])["age_class"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=EXPECTED_YEARS, columns=EXPECTED_SEXES, fill_value=0)
)

archive_summary = pd.DataFrame({
    "year": EXPECTED_YEARS,
    "female_files": files_by_sex_year["f"].to_numpy(),
    "male_files": files_by_sex_year["m"].to_numpy(),
    "female_age_classes": age_classes_by_sex_year["f"].to_numpy(),
    "male_age_classes": age_classes_by_sex_year["m"].to_numpy(),
})

archive_summary["total_files"] = (
    archive_summary["female_files"] + archive_summary["male_files"]
)

archive_summary

,year,female_files,male_files,female_age_classes,male_age_classes,total_files
0,2000,18,18,18,18,36
1,2001,18,18,18,18,36
2,2002,18,18,18,18,36
3,2003,18,18,18,18,36
4,2004,18,18,18,18,36
5,2005,18,18,18,18,36
6,2006,18,18,18,18,36
7,2007,18,18,18,18,36
8,2008,18,18,18,18,36
9,2009,18,18,18,18,36


### 4.1 Identificação de combinações ausentes

A lista esperada é construída explicitamente e comparada ao inventário observado.

In [5]:
expected_records = []

for year in EXPECTED_YEARS:
    for sex in EXPECTED_SEXES:
        for age_class in EXPECTED_AGE_CLASSES:
            expected_records.append({
                "year": year,
                "sex": sex,
                "age_class": age_class,
                "filename": f"bra_{sex}_{age_class}_{year}.tif",
            })

expected_inventory = pd.DataFrame(expected_records)

observed_keys = set(
    archive_inventory[["year", "sex", "age_class"]]
    .itertuples(index=False, name=None)
)

expected_keys = set(
    expected_inventory[["year", "sex", "age_class"]]
    .itertuples(index=False, name=None)
)

missing_keys = sorted(expected_keys - observed_keys)
unexpected_keys = sorted(observed_keys - expected_keys)

print(f"Combinações esperadas  : {len(expected_keys)}")
print(f"Combinações encontradas: {len(observed_keys)}")
print(f"Combinações ausentes   : {len(missing_keys)}")
print(f"Combinações inesperadas: {len(unexpected_keys)}")

if missing_keys:
    print("\nPrimeiras ausências:")
    print(missing_keys[:20])

Combinações esperadas  : 756
Combinações encontradas: 756
Combinações ausentes   : 0
Combinações inesperadas: 0


## 5. Verificação da coleção de idade e sexo no Earth Engine para 2020

A coleção pública do Earth Engine é usada como referência complementar para verificar a estrutura do produto em 2020. Ela não substitui o arquivo histórico 2000–2020.

In [6]:

age_sex_collection = (
    ee.ImageCollection(WORLDPOP_GEE_AGESEX)
    .filter(ee.Filter.eq("country", "BRA"))
)

gee_image_count = age_sex_collection.size().getInfo()

gee_years = sorted({
    int(year)
    for year in age_sex_collection.aggregate_array("year").getInfo()
})

print(f"Imagens do Brasil no GEE: {gee_image_count}")
print(f"Anos disponíveis        : {gee_years}")

Imagens do Brasil no GEE: 1
Anos disponíveis        : [2020]


In [7]:
image_2020 = ee.Image(
    age_sex_collection
    .filter(ee.Filter.eq("year", 2020))
    .first()
)

bands_2020 = image_2020.bandNames().getInfo()

female_bands = sorted(
    [b for b in bands_2020 if b.startswith("F_")],
    key=lambda x: int(x.split("_")[1])
)

male_bands = sorted(
    [b for b in bands_2020 if b.startswith("M_")],
    key=lambda x: int(x.split("_")[1])
)

female_ages_gee = [int(b.split("_")[1]) for b in female_bands]
male_ages_gee = [int(b.split("_")[1]) for b in male_bands]

print(f"Número total de bandas : {len(bands_2020)}")
print(f"Bandas femininas       : {len(female_bands)}")
print(f"Bandas masculinas      : {len(male_bands)}")
print(f"Classes femininas      : {female_ages_gee}")
print(f"Classes masculinas     : {male_ages_gee}")

Número total de bandas : 37
Bandas femininas       : 18
Bandas masculinas      : 18
Classes femininas      : [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]
Classes masculinas     : [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80]


In [8]:
reference_band = (
    "population"
    if "population" in bands_2020
    else female_bands[0]
)

projection = image_2020.select(reference_band).projection()

gee_crs = projection.crs().getInfo()
gee_scale = projection.nominalScale().getInfo()
gee_transform = projection.transform().getInfo()

print(f"Banda de referência : {reference_band}")
print(f"CRS                : {gee_crs}")
print(f"Resolução nominal  : {gee_scale:.3f} m")
print(f"Transformação      : {gee_transform}")

Banda de referência : population
CRS                : EPSG:4326
Resolução nominal  : 92.766 m
Transformação      : PARAM_MT["Affine", 
  PARAMETER["num_row", 3], 
  PARAMETER["num_col", 3], 
  PARAMETER["elt_0_0", 0.0008333333300044304], 
  PARAMETER["elt_0_2", -73.989583022], 
  PARAMETER["elt_1_1", -0.0008333333300081173], 
  PARAMETER["elt_1_2", 5.264583514]]


## 6. Correspondência entre arquivo histórico e Earth Engine

As classes etárias do Earth Engine em 2020 são comparadas às classes esperadas no arquivo Global1.

In [9]:
crosswalk_check = pd.Series({
    "Classes etárias esperadas": len(EXPECTED_AGE_CLASSES),
    "Classes femininas no GEE": len(female_ages_gee),
    "Classes masculinas no GEE": len(male_ages_gee),
    "Classes femininas compatíveis": female_ages_gee == EXPECTED_AGE_CLASSES,
    "Classes masculinas compatíveis": male_ages_gee == EXPECTED_AGE_CLASSES,
    "Banda population disponível": "population" in bands_2020,
    "CRS": gee_crs,
    "Resolução nominal (m)": gee_scale,
})

crosswalk_check

,0
Classes etárias esperadas,18
Classes femininas no GEE,18
Classes masculinas no GEE,18
Classes femininas compatíveis,True
Classes masculinas compatíveis,True
Banda population disponível,True
CRS,EPSG:4326
Resolução nominal (m),92.766242


## 7. Testes automatizados

Os testes validam apenas a disponibilidade e a estrutura da fonte.

In [12]:
assert SO32_LOGS.is_dir(), \
    "Diretório de logs do SO3.2 não encontrado."

assert (directory_status["http_status"] == 200).all(), \
    "Um ou mais diretórios anuais do WorldPop não estão acessíveis."

assert len(archive_inventory) == EXPECTED_TOTAL_FILES, \
    "Número inesperado de arquivos no inventário histórico."

assert len(missing_keys) == 0, \
    "Há combinações ano/sexo/classe etária ausentes."

assert len(unexpected_keys) == 0, \
    "Há combinações ano/sexo/classe etária inesperadas."

assert (archive_summary["female_files"] == EXPECTED_FILES_PER_SEX_YEAR).all(), \
    "Há ano com número inesperado de arquivos femininos."

assert (archive_summary["male_files"] == EXPECTED_FILES_PER_SEX_YEAR).all(), \
    "Há ano com número inesperado de arquivos masculinos."

assert gee_years == [2020], \
    "A disponibilidade temporal observada no GEE difere do esperado."

assert female_ages_gee == EXPECTED_AGE_CLASSES, \
    "As classes femininas do GEE diferem do arquivo histórico esperado."

assert male_ages_gee == EXPECTED_AGE_CLASSES, \
    "As classes masculinas do GEE diferem do arquivo histórico esperado."

assert gee_crs == "EPSG:4326", \
    "O CRS observado no GEE difere do esperado."

assert gee_image_count == 1, \
    "Número inesperado de imagens brasileiras no GEE para a coleção age/sex."

assert len(bands_2020) == 37, \
    "Número total de bandas do produto age/sex difere do esperado."

assert "population" in bands_2020, \
    "A banda population não está presente no produto 2020."

print(
    "Controles de disponibilidade e estrutura "
    "da população desagregada por sexo atendidos."
)

Controles de disponibilidade e estrutura da população desagregada por sexo atendidos.


## 8. Registro dos produtos de QA/QC

São gravados no diretório de logs:

- `so32_03_worldpop_sex_archive_inventory.csv`;
- `so32_03_worldpop_sex_archive_summary.csv`;
- `so32_03_worldpop_sex_gee2020_bands.csv`.

Os GeoTIFFs históricos não são copiados para o Google Drive nesta etapa.

In [13]:
archive_inventory_output = (
    SO32_LOGS / "so32_03_worldpop_sex_archive_inventory.csv"
)

archive_summary_output = (
    SO32_LOGS / "so32_03_worldpop_sex_archive_summary.csv"
)

gee_bands_output = (
    SO32_LOGS / "so32_03_worldpop_sex_gee2020_bands.csv"
)

archive_inventory.to_csv(
    archive_inventory_output,
    index=False
)

archive_summary.to_csv(
    archive_summary_output,
    index=False
)

gee_bands_table = pd.DataFrame({
    "band": bands_2020,
    "group": [
        "female" if band.startswith("F_")
        else "male" if band.startswith("M_")
        else "other"
        for band in bands_2020
    ],
})

gee_bands_table.to_csv(
    gee_bands_output,
    index=False
)

print(f"Inventário histórico : {archive_inventory_output}")
print(f"Resumo anual         : {archive_summary_output}")
print(f"Bandas GEE 2020      : {gee_bands_output}")

Inventário histórico : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_03_worldpop_sex_archive_inventory.csv
Resumo anual         : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_03_worldpop_sex_archive_summary.csv
Bandas GEE 2020      : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_03_worldpop_sex_gee2020_bands.csv


## 9. Síntese da validação

O inventário da série histórica WorldPop Global1 desagregada por sexo
confirmou a disponibilidade completa dos dados para o Brasil entre
2000 e 2020.

Foram identificados 756 GeoTIFFs, correspondentes a 21 anos,
18 classes etárias femininas e 18 classes etárias masculinas por ano.
Não foram observadas combinações ausentes ou inesperadas.

A coleção `WorldPop/GP/100m/pop_age_sex` disponível no Google Earth
Engine para 2020 apresenta 37 bandas: a banda de população total
(`population`), 18 bandas femininas e 18 bandas masculinas.

As classes etárias femininas e masculinas observadas no Earth Engine
correspondem integralmente às 18 classes identificadas na série
histórica:

`0, 1, 5, 10, 15, ..., 80`.

O produto utiliza `EPSG:4326` e apresenta resolução nominal de
aproximadamente 92,77 m.

### Situação da entrada

A série WorldPop desagregada por sexo é considerada tecnicamente
adequada para a construção das populações feminina e masculina
utilizadas no SO3.2.

Para os anos 2021–2023 será adotada, conforme o procedimento definido
para o PRAIS 4, a replicação dos valores de 2020.